In [0]:

%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS retail_project1_ext
URL 'abfss://retaildata@retailprojec1data.dfs.core.windows.net/'
WITH (
    STORAGE CREDENTIAL retail_project_cred
);

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS retail_project1_ext2
URL 'abfss://bronze@retailprojec1data.dfs.core.windows.net/'
WITH(
    STORAGE CREDENTIAL retail_project_cred
);

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_storage = 'abfss://bronze@retailprojec1data.dfs.core.windows.net/'

# parquet to delta 
print("Converting products Parquet to Delta...")
productsParquet = spark.read.parquet(f"{bronze_storage}/products/dbo.products.parquet")

productsBronze = productsParquet \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source_system", lit("azure_sql_via_adf"))

productsBronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{bronze_storage}/products_delta/")


In [0]:
# --- Customers ---
print("Converting customers Parquet to Delta...")
customersParquet = spark.read.parquet(f"{bronze_storage}/customers/dbo.customers.parquet")

customersBronze = customersParquet \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source_system", lit("azure_sql_via_adf"))

customersBronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{bronze_storage}/customers_delta/")

print(f"  ✅ {customersBronze.count()} rows in bronze.customers_delta")

In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS retail")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS retail.bronze_products
USING DELTA
LOCATION 'abfss://bronze@retailprojec1data.dfs.core.windows.net/products_delta/'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.bronze_customers
USING DELTA
LOCATION 'abfss://bronze@retailprojec1data.dfs.core.windows.net/customers_delta/'
""")